In [1]:
import pandas as pd
import numpy as np
# import cugraph
import glob
import matplotlib.pyplot as plt 
from multiprocessing import Pool
from ast import literal_eval
import swifter
import networkx as nx
import netwulf as nw
from netwulf import visualize
import netwulf as nw
from pandarallel import pandarallel
# import cugraph

import zipfile
import pandas as pd
import os
import glob
from functions import *
from ast import literal_eval
from multiprocessing import Pool
import numpy as np
from collections import Counter

from ast import literal_eval
import tldextract

In [2]:
pandarallel.initialize(progress_bar=True, nb_workers=24)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [3]:
ex = pd.read_pickle("simulations_group.pkl", compression="bz2")

In [3]:
steps=400
dorm_rate = 0.01

def simulate_diffusion_dormancy(p):
    A, inf_og, deg = p
    inf = inf_og.copy()
    inf_count = np.zeros(steps+1)
    inf_count[0] = inf.sum()
    
    dorm   = (np.random.rand(len(inf)) < dorm_rate)*inf # only ones that are infected are excluded
    
    for i in range(steps):
        dorm_delta   = (np.random.rand(len(inf)) < dorm_rate)*inf
        dorm += dorm_delta
        viable = inf * (1-dorm)
        
        cands = (A * viable).sum(axis=1) # for those viable to be bconverted.\
        prop = cands / deg 
        prop_adj = prop * (1-inf)
        delta = np.random.rand(len(inf)) < (prop_adj*0.05)
        inf += delta
        inf_count[i+1] = inf.sum()
    return inf_count

In [5]:
%%time
ex["ts_dorm"] = ex.params.parallel_apply(simulate_diffusion_dormancy)

CPU times: user 2.33 s, sys: 6.45 s, total: 8.78 s
Wall time: 1h 48min 8s


In [6]:
ex.to_pickle("simulations_group.pkl", compression="bz2")

In [7]:
ex = [0]

In [8]:
ex2 = pd.read_pickle("indv_hashtag_simulations.pkl", compression="bz2")

In [9]:
%%time
ex2["ts_dorm"] = ex2.params.parallel_apply(simulate_diffusion_dormancy)

CPU times: user 33.8 s, sys: 17.3 s, total: 51 s
Wall time: 1d 18h 33min 19s


In [10]:
ex2.to_pickle("indv_hashtag_simulations.pkl", compression="bz2")